<a href="https://colab.research.google.com/github/webathonic/Women_in_Geospatial_mini_projects/blob/master/rainfall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install pyhomogeneity

"""
CHIRPS Fitness-for-Use Audit Pipeline
======================================
Implements the four-step pre-analysis validation framework for CHIRPS data
in Nigerian agricultural policy contexts.

Paper: "Reassessing Seasonal Rainfall Trends in Nigeria:
        Implications for Policy Use of CHIRPS Data under Climate Change"

Input:  Nigeria_CHIRPS_Analysis_Results.csv  (exported from Google Earth Engine)
Output: audit_report.csv, plots per state, console fitness-for-use decision

Requirements:
    pip install pandas numpy matplotlib seaborn scipy pyhomogeneity
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from pyhomogeneity import pettitt_test
import warnings
import os

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

CSV_PATH = "/content/drive/MyDrive/Nigeria_CHIRPS_Analysis_Results.csv"   # <-- change if needed
OUTPUT_DIR = "audit_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regional classification (expand as needed)
NORTH_STATES = ['Kano', 'Jigawa', 'Borno', 'Kebbi', 'Sokoto',
                'Zamfara', 'Yobe', 'Katsina', 'Bauchi', 'Gombe']
SOUTH_STATES = ['Lagos', 'Rivers', 'Delta', 'Cross River',
                'Akwa Ibom', 'Ondo', 'Ogun', 'Bayelsa', 'Edo', 'Anambra']

# Thresholds
BREAK_P_THRESHOLD     = 0.05   # Pettitt test significance level
ONSET_TOL_DAYS        = 10     # Flag if onset discrepancy > 10 days
ONSET_RED_DAYS        = 15     # Hard fail if onset discrepancy > 15 days
BIAS_AMBER_PCT        = 10     # % bias → amber flag
BIAS_RED_PCT          = 20     # % bias → red flag

# Northern wet season months (unimodal)
NORTH_WET_MONTHS      = list(range(5, 10))   # May–Sep
# Southern first and second rainy seasons
SOUTH_S1_MONTHS       = list(range(3, 8))    # Mar–Jul
SOUTH_S2_MONTHS       = list(range(8, 12))   # Aug–Nov

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def traffic_light(flag):
    icons = {"GREEN": "🟢", "AMBER": "🟡", "RED": "🔴"}
    return f"{icons.get(flag, '?')} {flag}"


def get_region(state):
    if state in NORTH_STATES:
        return "NORTH"
    elif state in SOUTH_STATES:
        return "SOUTH"
    return "MIDDLE_BELT"


def annual_totals(df, state):
    sub = df[df['ADM1_NAME'] == state].copy()
    return sub.groupby('year')['rainfall_mm'].sum().reset_index()


def seasonal_totals(df, state, months):
    sub = df[(df['ADM1_NAME'] == state) & (df['month'].isin(months))].copy()
    return sub.groupby('year')['rainfall_mm'].sum().reset_index()


# ─────────────────────────────────────────────
# STEP 1 — TEMPORAL STATIONARITY (Pettitt Test)
# ─────────────────────────────────────────────

def step1_stationarity(df, state, region):
    """
    Detect structural breaks in annual or seasonal rainfall time series.
    Returns dict with: flag, breakpoint_year, p_value, interpretation
    """
    if region == "NORTH":
        series_df = seasonal_totals(df, state, NORTH_WET_MONTHS)
        season_label = "May–Sep (wet season)"
    else:
        series_df = annual_totals(df, state)
        season_label = "Annual total"

    series = series_df['rainfall_mm'].dropna()
    years  = series_df['year'].values

    result = pettitt_test(series)
    cp_idx = result.cp                         # index of breakpoint
    p_val  = result.p

    flag = "GREEN"
    bp_year = None
    if p_val < BREAK_P_THRESHOLD:
        flag    = "RED"
        bp_year = int(years[cp_idx]) if cp_idx < len(years) else None

    # Pre / post means
    pre_mean  = series.iloc[:cp_idx].mean()  if cp_idx else np.nan
    post_mean = series.iloc[cp_idx:].mean()  if cp_idx else np.nan

    return {
        "step": "1_stationarity",
        "state": state,
        "region": region,
        "flag": flag,
        "breakpoint_year": bp_year,
        "p_value": round(p_val, 4),
        "pre_break_mean_mm": round(pre_mean, 1) if not np.isnan(pre_mean) else None,
        "post_break_mean_mm": round(post_mean, 1) if not np.isnan(post_mean) else None,
        "series_label": season_label,
        "series": series,
        "years": years
    }


# ─────────────────────────────────────────────
# STEP 2 — SEASONAL CALENDAR VALIDATION
# ─────────────────────────────────────────────

def estimate_onset(monthly_series, year, region):
    """
    Simplified onset estimator: first month where 3-month rolling rainfall
    exceeds threshold, consistent with Walter (1967) spirit.
    Returns estimated onset month number.
    """
    sub = monthly_series[monthly_series['year'] == year].sort_values('month')
    if sub.empty:
        return np.nan

    if region == "NORTH":
        # Look for onset in Apr–Jul window
        window = sub[sub['month'].between(4, 7)]
        cumsum  = 0
        for _, row in window.iterrows():
            cumsum += row['rainfall_mm']
            if cumsum >= 20:
                return int(row['month'])
    else:
        # South: first sustained month in Mar–Jun
        window = sub[sub['month'].between(3, 6)]
        for _, row in window.iterrows():
            if row['rainfall_mm'] >= 30:
                return int(row['month'])
    return np.nan


def step2_calendar(df, state, region):
    """
    Compare CHIRPS-derived onset in early vs late periods.
    Proxy for gauge comparison: we compare 1981-2002 mean onset vs 2003-2023.
    A shift in onset between periods acts as a flag of calendar drift.
    """
    monthly = df[df['ADM1_NAME'] == state][['year', 'month', 'rainfall_mm']].copy()

    early_onsets = [estimate_onset(monthly, y, region) for y in range(1981, 2003)]
    late_onsets  = [estimate_onset(monthly, y, region) for y in range(2003, 2024)]

    early_onsets = [o for o in early_onsets if not np.isnan(o)]
    late_onsets  = [o for o in late_onsets  if not np.isnan(o)]

    if not early_onsets or not late_onsets:
        return {"step": "2_calendar", "state": state, "flag": "AMBER",
                "note": "Insufficient data for onset estimation"}

    early_mean = np.mean(early_onsets)
    late_mean  = np.mean(late_onsets)
    shift_days = abs(late_mean - early_mean) * 30  # months → approx days

    if shift_days > ONSET_RED_DAYS:
        flag = "RED"
    elif shift_days > ONSET_TOL_DAYS:
        flag = "AMBER"
    else:
        flag = "GREEN"

    return {
        "step": "2_calendar",
        "state": state,
        "region": region,
        "flag": flag,
        "early_period_mean_onset_month": round(early_mean, 2),
        "late_period_mean_onset_month": round(late_mean, 2),
        "approx_shift_days": round(shift_days, 1),
        "direction": "Later" if late_mean > early_mean else "Earlier"
    }


# ─────────────────────────────────────────────
# STEP 3 — SPATIAL RESOLUTION ADEQUACY
# ─────────────────────────────────────────────

def step3_resolution(state, region):
    """
    Rule-based assessment of whether 5km CHIRPS resolution is adequate
    for the region and application (crop planning).
    Based on rainfall regime type.
    """
    if region == "SOUTH":
        flag = "AMBER"
        note = ("Convective, spatially heterogeneous rainfall in the south. "
                "5km may mask intra-state variability. "
                "Recommend downscaling for district-level use.")
    elif region == "NORTH":
        flag = "GREEN"
        note = ("Stratiform, large-scale rainfall in northern savanna zone. "
                "5km generally adequate for seasonal monitoring.")
    else:
        flag = "AMBER"
        note = ("Middle Belt: mixed convective/stratiform regime. "
                "5km adequate for national monitoring but verify for local use.")

    return {
        "step": "3_resolution",
        "state": state,
        "region": region,
        "flag": flag,
        "chirps_resolution_km": 5,
        "note": note
    }


# ─────────────────────────────────────────────
# STEP 4 — BIAS QUANTIFICATION
# ─────────────────────────────────────────────

def step4_bias(df, state, region):
    """
    Estimates trend-based bias by comparing observed rainfall trend direction
    vs. documented climate expectations.
    Full implementation requires gauge station CSV — here we use Mann-Kendall
    trend test and flag anomalous trend behaviour.
    """
    if region == "NORTH":
        series_df = seasonal_totals(df, state, NORTH_WET_MONTHS)
    else:
        series_df = annual_totals(df, state)

    series = series_df['rainfall_mm'].dropna().values
    if len(series) < 10:
        return {"step": "4_bias", "state": state, "flag": "AMBER",
                "note": "Insufficient data for bias quantification"}

    # Mann-Kendall trend
    slope, intercept, r, p, se = stats.linregress(range(len(series)), series)

    # Coefficient of variation — high CV in north signals erratic seasons
    cv = (series.std() / series.mean()) * 100

    # Bias proxy: % change from first decade to last decade mean
    first_dec = series[:10].mean()
    last_dec  = series[-10:].mean()
    pct_change = ((last_dec - first_dec) / first_dec) * 100

    if abs(pct_change) > BIAS_RED_PCT:
        flag = "RED"
    elif abs(pct_change) > BIAS_AMBER_PCT:
        flag = "AMBER"
    else:
        flag = "GREEN"

    return {
        "step": "4_bias",
        "state": state,
        "region": region,
        "flag": flag,
        "trend_slope_mm_per_year": round(slope, 3),
        "trend_p_value": round(p, 4),
        "first_decade_mean_mm": round(first_dec, 1),
        "last_decade_mean_mm": round(last_dec, 1),
        "pct_change_first_to_last_decade": round(pct_change, 1),
        "coeff_variation_pct": round(cv, 1)
    }


# ─────────────────────────────────────────────
# COMPOSITE FITNESS-FOR-USE DECISION
# ─────────────────────────────────────────────

FLAG_RANK = {"GREEN": 0, "AMBER": 1, "RED": 2}

def composite_flag(flags):
    return max(flags, key=lambda f: FLAG_RANK.get(f, 0))


def fitness_decision(results):
    flags = [r["flag"] for r in results]
    final = composite_flag(flags)
    messages = {
        "GREEN": "✅ CHIRPS data is fit for use in standard seasonal analysis.",
        "AMBER": "⚠️  Use with caution. Results should include uncertainty bounds.",
        "RED":   "🚫 Recalibrate before policy use. Baseline may be unrepresentative."
    }
    return final, messages[final]


# ─────────────────────────────────────────────
# VISUALISATION
# ─────────────────────────────────────────────

def plot_state_audit(df, state, results, step1_data):
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"CHIRPS Fitness-for-Use Audit — {state}", fontsize=15, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

    region = get_region(state)
    color_map = {"GREEN": "#2ecc71", "AMBER": "#f39c12", "RED": "#e74c3c"}

    # — Plot 1: Annual / seasonal time series with breakpoint —
    ax1 = fig.add_subplot(gs[0, :])
    series = step1_data["series"]
    years  = step1_data["years"]
    bp_year = step1_data.get("breakpoint_year")

    ax1.bar(years, series, color='steelblue', alpha=0.6, label='Annual/Seasonal total')
    z = np.polyfit(years, series, 1)
    ax1.plot(years, np.poly1d(z)(years), 'r--', linewidth=1.5, label='Trend')

    if bp_year:
        ax1.axvline(bp_year, color='orange', linewidth=2, linestyle='--',
                    label=f'Breakpoint ({bp_year})')
        ax1.axvspan(years[0], bp_year, alpha=0.05, color='blue')
        ax1.axvspan(bp_year, years[-1], alpha=0.05, color='red')

    ax1.set_title(f"Rainfall Time Series — {step1_data['series_label']}", fontsize=11)
    ax1.set_ylabel("Rainfall (mm)")
    ax1.legend(fontsize=9)
    ax1.grid(axis='y', alpha=0.3)

    # — Plot 2: Monthly climatology comparison (early vs late) —
    ax2 = fig.add_subplot(gs[1, 0])
    monthly = df[df['ADM1_NAME'] == state].copy()
    early = monthly[monthly['year'] <= 2002].groupby('month')['rainfall_mm'].mean()
    late  = monthly[monthly['year'] > 2002].groupby('month')['rainfall_mm'].mean()
    month_names = ['J','F','M','A','M','J','J','A','S','O','N','D']
    x = np.arange(1, 13)
    ax2.plot(x, [early.get(m, 0) for m in x], 'b-o', markersize=4, label='1981–2002', linewidth=1.5)
    ax2.plot(x, [late.get(m, 0)  for m in x], 'r-o', markersize=4, label='2003–2023', linewidth=1.5)
    ax2.set_xticks(x); ax2.set_xticklabels(month_names)
    ax2.set_title("Monthly Climatology: Early vs Late Period", fontsize=10)
    ax2.set_ylabel("Mean Monthly Rainfall (mm)")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    # — Plot 3: Audit scorecard —
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.axis('off')
    step_labels = ["Step 1\nStationarity", "Step 2\nCalendar", "Step 3\nResolution", "Step 4\nBias"]
    flags = [r["flag"] for r in results]
    final_flag, final_msg = fitness_decision(results)

    for i, (label, flag) in enumerate(zip(step_labels, flags)):
        y_pos = 0.82 - i * 0.18
        ax3.add_patch(plt.Rectangle((0.05, y_pos - 0.06), 0.9, 0.13,
                                    color=color_map[flag], alpha=0.25, transform=ax3.transAxes))
        ax3.text(0.12, y_pos, label, transform=ax3.transAxes, fontsize=9,
                 va='center', ha='left')
        ax3.text(0.75, y_pos, traffic_light(flag), transform=ax3.transAxes,
                 fontsize=10, va='center', ha='center')

    ax3.add_patch(plt.Rectangle((0.05, 0.0), 0.9, 0.14,
                                color=color_map[final_flag], alpha=0.4, transform=ax3.transAxes))
    ax3.text(0.5, 0.07, f"OVERALL: {final_flag}", transform=ax3.transAxes,
             fontsize=11, fontweight='bold', va='center', ha='center',
             color=color_map[final_flag])

    ax3.set_title("Audit Scorecard", fontsize=10)

    plt.savefig(os.path.join(OUTPUT_DIR, f"audit_{state.replace(' ', '_')}.png"),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  → Plot saved: audit_{state.replace(' ', '_')}.png")


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(csv_path):
    print("\n" + "="*60)
    print("  CHIRPS FITNESS-FOR-USE AUDIT PIPELINE")
    print("="*60)

    # Load data
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Standardise column names (GEE export may vary)
    rename_map = {'mean': 'rainfall_mm', 'ADM1_NAME': 'ADM1_NAME'}
    df = df.rename(columns={c: rename_map[c] for c in rename_map if c in df.columns})

    # Ensure numeric types
    df['year']         = df['year'].astype(int)
    df['month']        = df['month'].astype(int)
    df['rainfall_mm']  = pd.to_numeric(df['rainfall_mm'], errors='coerce')
    df                 = df.dropna(subset=['rainfall_mm'])

    states = df['ADM1_NAME'].unique()
    print(f"\nStates detected: {list(states)}")

    all_results = []
    report_rows = []

    for state in states:
        region = get_region(state)
        print(f"\n{'─'*50}")
        print(f"  STATE: {state}  |  Region: {region}")
        print(f"{'─'*50}")

        # Run four steps
        r1 = step1_stationarity(df, state, region)
        r2 = step2_calendar(df, state, region)
        r3 = step3_resolution(state, region)
        r4 = step4_bias(df, state, region)

        step_results = [r1, r2, r3, r4]

        # Print step results
        for r in step_results:
            print(f"  {r['step'].upper():25s}  {traffic_light(r['flag'])}")
            for k, v in r.items():
                if k not in ('step', 'state', 'region', 'flag', 'series', 'years'):
                    print(f"    {k:35s}: {v}")

        final_flag, final_msg = fitness_decision(step_results)
        print(f"\n  COMPOSITE DECISION: {final_msg}\n")

        # Plot
        plot_state_audit(df, state, step_results, r1)

        # Collect for report
        row = {
            "state": state,
            "region": region,
            "step1_flag": r1["flag"],
            "step1_breakpoint_year": r1.get("breakpoint_year"),
            "step1_p_value": r1.get("p_value"),
            "step2_flag": r2["flag"],
            "step2_approx_shift_days": r2.get("approx_shift_days"),
            "step2_direction": r2.get("direction"),
            "step3_flag": r3["flag"],
            "step3_note": r3["note"],
            "step4_flag": r4["flag"],
            "step4_pct_change": r4.get("pct_change_first_to_last_decade"),
            "step4_trend_slope": r4.get("trend_slope_mm_per_year"),
            "overall_flag": final_flag,
            "decision": final_msg
        }
        report_rows.append(row)
        all_results.append(step_results)

    # Save report
    report_df = pd.DataFrame(report_rows)
    report_path = os.path.join(OUTPUT_DIR, "audit_report.csv")
    report_df.to_csv(report_path, index=False)
    print(f"\n{'='*60}")
    print(f"  Audit complete. Report saved to: {report_path}")
    print(f"  Plots saved in: {OUTPUT_DIR}/")
    print("="*60 + "\n")

    return report_df


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    report = run_pipeline(CSV_PATH)
    print(report[['state', 'region', 'step1_flag', 'step2_flag',
                  'step3_flag', 'step4_flag', 'overall_flag']].to_string(index=False))


  CHIRPS FITNESS-FOR-USE AUDIT PIPELINE

States detected: ['Gombe', 'Sokoto', 'Benue', 'Kano', 'Lagos', 'Niger', 'Oyo']

──────────────────────────────────────────────────
  STATE: Gombe  |  Region: NORTH
──────────────────────────────────────────────────
  1_STATIONARITY             🟢 GREEN
    breakpoint_year                    : None
    p_value                            : 0.198
    pre_break_mean_mm                  : 735.9
    post_break_mean_mm                 : 831.1
    series_label                       : May–Sep (wet season)
  2_CALENDAR                 🟢 GREEN
    early_period_mean_onset_month      : 4.27
    late_period_mean_onset_month       : 4.19
    approx_shift_days                  : 2.5
    direction                          : Earlier
  3_RESOLUTION               🟢 GREEN
    chirps_resolution_km               : 5
    note                               : Stratiform, large-scale rainfall in northern savanna zone. 5km generally adequate for seasonal monitoring.
  4_BI

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
